# Human-in-the-Loop 간단 예제

5장에서 배운 `HumanInTheLoopMiddleware`를 파일 작업 에이전트에 적용해 봅니다.

- `write_file`, `delete_file` → 승인 필요 (approve / edit / reject)
- `read_file` → 승인 불필요 (안전한 읽기 작업)

**모델**: API 키 없이 로컬 [Ollama](https://ollama.com) 모델(`llama3.1:8b`)을 사용합니다. tool calling을 지원하는 모델이라 HITL 흐름이 그대로 동작합니다. Claude API로 바꾸려면 모델 초기화 셀만 `init_chat_model("claude-sonnet-4-5")`로 교체하고 프로젝트 루트의 `.env`에 `ANTHROPIC_API_KEY`를 넣으면 됩니다.

**사전 준비**:
1. Ollama 서버 실행 확인 (`ollama serve`, 이미 백그라운드로 떠 있다면 생략)
2. 모델 준비: `ollama pull llama3.1:8b` (약 4.9GB, 한 번만)

**실행**: 프로젝트 루트에서 `uv run jupyter lab examples/ch05-human-in-the-loop/hitl_example.ipynb`

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

## 1. 도구 정의

파일을 쓰고, 읽고, 지우는 3개의 도구를 정의합니다.

In [3]:
@tool
def write_file(filename: str, content: str) -> str:
    """파일에 내용을 작성합니다."""
    with open(filename, "w") as f:
        f.write(content)
    return f"File {filename} written successfully"


@tool
def read_file(filename: str) -> str:
    """파일에서 내용을 읽어옵니다."""
    try:
        with open(filename, "r") as f:
            return f.read()
    except FileNotFoundError:
        return f"File {filename} not found"


@tool
def delete_file(filename: str) -> str:
    """파일을 삭제합니다."""
    import os

    try:
        os.remove(filename)
        return f"File {filename} deleted successfully"
    except FileNotFoundError:
        return f"File {filename} not found"

## 2. HITL 미들웨어와 함께 에이전트 생성

`write_file`, `delete_file`은 승인이 필요하고, `read_file`은 바로 실행되도록 설정합니다.
체크포인터(`InMemorySaver`)가 없으면 interrupt 이후 재개할 수 없으므로 필수입니다.

In [4]:
model = init_chat_model("ollama:llama3.1:8b", temperature=0)
# Claude API를 쓰려면 위 줄 대신:
# model = init_chat_model("claude-sonnet-4-5")

agent = create_agent(
    model=model,
    tools=[write_file, read_file, delete_file],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": True,   # approve / edit / reject 모두 허용
                "delete_file": True,
                "read_file": False,   # 승인 불필요
            },
            description_prefix="Tool execution pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
)

print("HITL 에이전트 생성 완료")

HITL 에이전트 생성 완료


## 3. Approve — 그대로 승인

`test.txt` 파일 쓰기를 요청하면 interrupt가 발생합니다. 어떤 도구가, 어떤 인수로 대기 중인지 확인해 봅니다.

In [5]:
config = {"configurable": {"thread_id": "thread_approve"}}

result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Write 'Hello World' to a file called test.txt"}
        ]
    },
    config=config,
)

if "__interrupt__" in result:
    action = result["__interrupt__"][0].value["action_requests"][0]
    print(f"승인 대기 중 → 도구: {action['name']}, 인수: {action['args']}")
else:
    print("interrupt가 발생하지 않았습니다")

승인 대기 중 → 도구: write_file, 인수: {'content': 'Hello World', 'filename': 'test.txt'}


In [5]:
# 그대로 승인 → 실제로 파일이 생성됩니다
result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,
)

print(result["messages"][-1].content)

with open("test.txt") as f:
    print("\n파일 내용:", f.read())

The `write_file` tool was used with the specified parameters, and the output indicates that the file has been created successfully.

파일 내용: Hello World


## 4. Edit — 인수를 수정해서 승인

이번엔 `original.txt`로 쓰려는 요청을 가로채서, 파일명과 내용을 바꿔 실행합니다.

In [6]:
config_edit = {"configurable": {"thread_id": "thread_edit"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Write 'Original content' to original.txt"}]},
    config=config_edit,
)

if "__interrupt__" in result:
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "write_file",
                            "args": {
                                "filename": "modified.txt",  # 파일명 변경
                                "content": "Modified content",  # 내용 변경
                            },
                        },
                    }
                ]
            }
        ),
        config=config_edit,
    )

with open("modified.txt") as f:
    print("modified.txt 내용:", f.read())

modified.txt 내용: Modified content


## 5. Reject — 거부하고 피드백 제공

`test.txt` 삭제 요청을 거부하고, 이유를 피드백으로 전달합니다. 에이전트가 이 피드백을 받아 어떻게 반응하는지 확인해 봅니다.

In [7]:
config_reject = {"configurable": {"thread_id": "thread_reject"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Delete the file test.txt"}]},
    config=config_reject,
)

if "__interrupt__" in result:
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "reject",
                        "message": "이 파일은 중요한 데이터를 담고 있어 삭제할 수 없습니다. 먼저 백업하세요.",
                    }
                ]
            }
        ),
        config=config_reject,
    )

print(result["messages"][-1].content)

{"name": "backup_file", "parameters": {"filename":"test.txt"}}


> **로컬 모델 관찰 포인트**: `llama3.1:8b`로 실행하면 위 출력이 `{"name": "backup_file", ...}` 같은 원문 JSON 텍스트로 나올 수 있습니다. reject 흐름 자체(interrupt → ToolMessage 합성 → 재개)는 정상 동작한 것이고, 모델이 "백업 먼저 하라"는 피드백을 받아 존재하지 않는 `backup_file` 도구를 스스로 지어내려다 실패한 것뿐입니다. 8B급 로컬 모델은 이런 상황에서 tool-call 형식을 깔끔하게 못 지키는 경우가 있습니다 — 5장에서 본 Claude 예제와 비교해보면 모델 크기·품질이 HITL 경험에 어떻게 영향을 주는지 체감할 수 있습니다.

## 6. 정리

실습 중 생성된 테스트 파일을 삭제합니다.

In [8]:
import os

for f in ["test.txt", "modified.txt", "original.txt"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"삭제됨: {f}")

삭제됨: modified.txt
